# Generative AI: Assignment 1

**Total Marks:** 100 | **Due Date:** 14th September, 9PM

This notebook covers:
- **Part 1:** Topic Detection & Summarization of BBC News Articles (45 marks)
- **Part 2:** Job Postings Analysis - Role Categorization & Requirements Extraction (55 marks)

## Initial Setup
### Set API key for OpenRouter
Click [here](https://openrouter.ai/keys) to create an API key, if not already created.

Browse available Llama models (and pricing/free tiers) at https://openrouter.ai/models.

In [2]:
pip install pandas

  Using cached pandas-3.0.5-cp313-cp313-win_amd64.whl.metadata (19 kB)
  Using cached numpy-2.5.3-cp313-cp313-win_amd64.whl.metadata (6.6 kB)
  Using cached tzdata-2026.4-py2.py3-none-any.whl.metadata (1.4 kB)
Using cached pandas-3.0.5-cp313-cp313-win_amd64.whl (9.8 MB)
Using cached numpy-2.5.3-cp313-cp313-win_amd64.whl (12.6 MB)
Using cached tzdata-2026.4-py2.py3-none-any.whl (347 kB)

   ---------------------------------------- 0/3 [tzdata]
   ---------------------------------------- 0/3 [tzdata]
   ---------------------------------------- 0/3 [tzdata]
   ---------------------------------------- 0/3 [tzdata]
   ---------------------------------------- 0/3 [tzdata]
   ---------------------------------------- 0/3 [tzdata]
   ---------------------------------------- 0/3 [tzdata]
   ---------------------------------------- 0/3 [tzdata]
   ---------------------------------------- 0/3 [tzdata]
   ------------- -------------------------- 1/3 [numpy]
   ------------- ------------------------

In [1]:
import os, json, re, getpass
import pandas as pd
from dotenv import load_dotenv

load_dotenv(override=True)

True

In [4]:
pip install dotenv

  Using cached dotenv-0.9.9-py2.py3-none-any.whl.metadata (279 bytes)
  Using cached python_dotenv-1.2.3-py3-none-any.whl.metadata (29 kB)
Using cached dotenv-0.9.9-py2.py3-none-any.whl (1.9 kB)
Using cached python_dotenv-1.2.3-py3-none-any.whl (22 kB)

   ---------------------------------------- 0/2 [python-dotenv]
   ---------------------------------------- 0/2 [python-dotenv]
   -------------------- ------------------- 1/2 [dotenv]
   ---------------------------------------- 2/2 [dotenv]

Note: you may need to restart the kernel to use updated packages.


In [2]:
if "OPENROUTER_API_KEY" not in os.environ:
    os.environ["OPENROUTER_API_KEY"] = getpass.getpass("OpenRouter API Key: ")

In [3]:
from langchain_openai import ChatOpenAI

model_name = "meta-llama/llama-3.1-8b-instruct"  # swap for e.g. ":free" suffix variant if available

llm = ChatOpenAI(
    model=model_name,
    base_url="https://openrouter.ai/api/v1",
    api_key=os.environ["OPENROUTER_API_KEY"],
    temperature=0,
)

In [ ]:
%pip install -U langchain langchain-openai

---
# Part 1: Topic Detection and Summarization of News Articles


## Step 1: Load the Dataset
Using the BBC News Full-Text dataset, limited to the first 30 articles.

In [4]:
news_df = pd.read_csv("bbc-news-data.csv", sep="\t")
news_df = news_df.head(30).reset_index(drop=True)
print(news_df.shape)
news_df.head()

(30, 4)


,category,filename,title,content
0,business,001.txt,Ad sales boost Time Warner profit,Quarterly profits at US media giant TimeWarne...
1,business,002.txt,Dollar gains on Greenspan speech,The dollar has hit its highest level against ...
2,business,003.txt,Yukos unit buyer faces loan claim,The owners of embattled Russian oil giant Yuk...
3,business,004.txt,High fuel prices hit BA's profits,British Airways has blamed high fuel prices f...
4,business,005.txt,Pernod takeover talk lifts Domecq,Shares in UK drinks and food firm Allied Dome...


## Step 2: Define the Topic Classification Task 

In [5]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

topic_categories = ["Business", "Entertainment", "Politics", "Sport", "Tech"]

classification_template = ChatPromptTemplate([
    ("system", "You are a news editor who classifies articles into a single topic category."),
    ("human", """Analyze the following news article and identify its topic as one of the following categories: {categories}.

Examples:
Article: "The prime minister announced new legislation in parliament today..." -> Politics
Article: "The striker scored a hat-trick to win the match for his team..." -> Sport

Article:
{article}

Return ONLY the single category label, nothing else."""),
])

classification_chain = classification_template | llm | StrOutputParser()

In [6]:
# Show this works for a sample datapoint
sample_article = news_df.loc[0, "content"]

sample_topic = classification_chain.invoke({
    "categories": ", ".join(topic_categories),
    "article": sample_article
}).strip()

print("Predicted Topic:", sample_topic)
print("Actual Category:", news_df.loc[0, "category"])

Predicted Topic: Business
Actual Category: business


## Step 3: Define the Summarization Task

In [7]:
summarization_template = ChatPromptTemplate([
    ("system", "You are a skilled news summarizer."),
    ("human", """Summarize the main points of the following news article in 2-3 sentences.
Capture the who/what/when/where/why as applicable, without adding personal commentary.

Article:
{article}"""),
])

summarization_chain = summarization_template | llm | StrOutputParser()

In [8]:
# Show this works for a sample datapoint
sample_summary = summarization_chain.invoke({"article": sample_article}).strip()
print(sample_summary)

Here is a 3-sentence summary of the article:

TimeWarner's quarterly profits rose 76% to $1.13 billion, driven by increased sales of high-speed internet connections and higher advertising revenue. The company's overall sales grew 2% to $11.1 billion, with its film division seeing a 27% profit decline due to box office flops. TimeWarner's full-year profit increased 27% to $3.36 billion, with the company projecting 5% operating earnings growth and higher revenue for 2005.


## Step 4: Key Entity Extraction


In [9]:
from pydantic import BaseModel, Field
from typing import List
from langchain_core.output_parsers import PydanticOutputParser

def to_model(obj, model_cls):
    """Safety net: pass through if already the right Pydantic type, else coerce a dict."""
    return obj if isinstance(obj, model_cls) else model_cls(**obj)

class KeyEntities(BaseModel):
    """Important entities mentioned in a news article."""
    people: List[str] = Field(description="Notable people mentioned in the article")
    organizations: List[str] = Field(description="Organizations/companies mentioned in the article")
    locations: List[str] = Field(description="Places/locations mentioned in the article")

entity_parser = PydanticOutputParser(pydantic_object=KeyEntities)

# Hand-written instructions with a filled-in example, instead of a raw JSON-schema
# dump: smaller models sometimes just echo the schema back verbatim otherwise.
entity_format_instructions = """Respond with ONLY a JSON object with exactly these keys, filled in with the actual values you found (not the schema, not descriptions):
{
  "people": ["Jane Smith", "John Doe"],
  "organizations": ["Acme Corp"],
  "locations": ["London"]
}
Use empty lists [] for any category with no matches. Output nothing except this JSON object."""

entity_template = ChatPromptTemplate([
    ("system", "You extract key named entities from news articles. Respond with valid JSON only, no extra commentary."),
    ("human", """From the article below, list the names of any important people, organizations, or places mentioned.

Article:
{article}

{format_instructions}"""),
]).partial(format_instructions=entity_format_instructions)

entity_chain = entity_template | llm | entity_parser


In [10]:
# Show this works for a sample datapoint
sample_entities = entity_chain.invoke({"article": sample_article})
sample_entities

KeyEntities(people=['Richard Parsons'], organizations=['TimeWarner', 'Google', 'AOL', 'SEC', 'Bertelsmann'], locations=['US', 'Germany'])

## Step 5: Update the DataFrame with Results
Combine all three tasks into a single structured chain (fewer LLM calls, more efficient) and apply it across the first 30 articles.

In [11]:
class ArticleAnalysis(BaseModel):
    """Structured analysis of a news article: topic, summary and key entities."""
    Detected_Topic: str = Field(description=f"The single best-fit topic, one of: {', '.join(topic_categories)}")
    Summary: str = Field(description="A concise 2-3 sentence summary of the article")
    Key_Entities: List[str] = Field(description="Important people, organizations, or locations mentioned in the article")

analysis_parser = PydanticOutputParser(pydantic_object=ArticleAnalysis)

analysis_format_instructions = """Respond with ONLY a JSON object with exactly these keys, filled in with the actual values you found (not the schema, not descriptions):
{
  "Detected_Topic": "Sport",
  "Summary": "A concise 2-3 sentence summary goes here.",
  "Key_Entities": ["Jane Smith", "Acme Corp", "London"]
}
Output nothing except this JSON object."""

analysis_template = ChatPromptTemplate([
    ("system", "You are an expert news analyst. Respond with valid JSON only, no extra commentary."),
    ("human", """Analyze the following news article and provide:
1. Its topic - one of: {categories}
2. A 2-3 sentence summary capturing the key points
3. A list of key entities (notable people, organizations, or locations)

Article:
{article}

{format_instructions}"""),
]).partial(format_instructions=analysis_format_instructions)

analysis_chain = analysis_template | llm | analysis_parser


In [14]:
results = []
for idx, row in news_df.iterrows():
    try:
        analysis = to_model(
            analysis_chain.invoke({
                "categories": ", ".join(topic_categories),
                "article": row["content"]
            }),
            ArticleAnalysis,
        )
        results.append(analysis.model_dump())
    except Exception as e:
        print(f"Row {idx} failed: {e}")
        results.append({"Detected_Topic": "Not specified", "Summary": "Not specified", "Key_Entities": []})

results_df = pd.DataFrame(results)
results_df.head()


,Detected_Topic,Summary,Key_Entities
0,Business,TimeWarner's quarterly profits jumped 76% to $...,"[Richard Parsons, TimeWarner, Google, AOL, War..."
1,Business,The dollar has reached its highest level again...,"[Alan Greenspan, Federal Reserve, Robert Sinch..."
2,Business,Yukos' owner Menatep Group is asking Rosneft t...,"[Menatep Group, Rosneft, Yugansk, Russia, Mikh..."
3,Business,British Airways reported a 40% drop in profits...,"[British Airways, Rod Eddington, Mike Powell, ..."
4,Business,Shares in Allied Domecq rose 4% on speculation...,"[Allied Domecq, Pernod Ricard, London, Wall St..."


In [15]:
# Final merged dataframe with all original and new columns together
news_final_df = pd.concat([news_df.reset_index(drop=True), results_df.reset_index(drop=True)], axis=1)
news_final_df.head()

,category,filename,title,content,Detected_Topic,Summary,Key_Entities
0,business,001.txt,Ad sales boost Time Warner profit,Quarterly profits at US media giant TimeWarne...,Business,TimeWarner's quarterly profits jumped 76% to $...,"[Richard Parsons, TimeWarner, Google, AOL, War..."
1,business,002.txt,Dollar gains on Greenspan speech,The dollar has hit its highest level against ...,Business,The dollar has reached its highest level again...,"[Alan Greenspan, Federal Reserve, Robert Sinch..."
2,business,003.txt,Yukos unit buyer faces loan claim,The owners of embattled Russian oil giant Yuk...,Business,Yukos' owner Menatep Group is asking Rosneft t...,"[Menatep Group, Rosneft, Yugansk, Russia, Mikh..."
3,business,004.txt,High fuel prices hit BA's profits,British Airways has blamed high fuel prices f...,Business,British Airways reported a 40% drop in profits...,"[British Airways, Rod Eddington, Mike Powell, ..."
4,business,005.txt,Pernod takeover talk lifts Domecq,Shares in UK drinks and food firm Allied Dome...,Business,Shares in Allied Domecq rose 4% on speculation...,"[Allied Domecq, Pernod Ricard, London, Wall St..."


In [16]:
news_final_df.to_csv("part1_news_analysis_results.csv", index=False)
news_final_df.shape

(30, 7)

---
# Part 2: Job Postings Analysis - Role Categorization and Requirements Extraction


## Step 1: Load the Dataset
Using the job postings dataset, limited to the first 25 postings.

In [17]:
jobs_df = pd.read_csv("job_title_des.csv")
jobs_df = jobs_df.rename(columns={"Job Title": "Job_Title", "Job Description": "Job_Description"})
jobs_df = jobs_df[["Job_Title", "Job_Description"]].head(25).reset_index(drop=True)
print(jobs_df.shape)
jobs_df.head()

(25, 2)


,Job_Title,Job_Description
0,Flutter Developer,We are looking for hire experts flutter develo...
1,Django Developer,PYTHON/DJANGO (Developer/Lead) - Job Code(PDJ ...
2,Machine Learning,"Data Scientist (Contractor)\r\n\r\nBangalore, ..."
3,iOS Developer,JOB DESCRIPTION:\r\n\r\nStrong framework outsi...
4,Full Stack Developer,job responsibility full stack engineer – react...


## Step 2: Define the Job Category Classification Task

In [18]:
job_categories = ["Technology/IT", "Finance", "Marketing", "Healthcare", "Education", "Sales", "Operations", "Other"]

job_classification_template = ChatPromptTemplate([
    ("system", "You are an HR specialist who categorizes job postings into a broad domain."),
    ("human", """Given the following job title and description, categorize the job into one of the following domains: {categories}.
If unsure, use "Other".

Job: {job_title}
Description: {job_description}

Return ONLY the single domain category label, nothing else."""),
])

job_classification_chain = job_classification_template | llm | StrOutputParser()

In [19]:
# Show this works for a sample datapoint
sample_job_title = jobs_df.loc[0, "Job_Title"]
sample_job_desc = jobs_df.loc[0, "Job_Description"]

sample_category = job_classification_chain.invoke({
    "categories": ", ".join(job_categories),
    "job_title": sample_job_title,
    "job_description": sample_job_desc
}).strip()

print("Job Title:", sample_job_title)
print("Predicted Category:", sample_category)

Job Title: Flutter Developer
Predicted Category: Technology/IT


## Step 3: Define the Requirements Extraction Task

In [20]:
class JobRequirements(BaseModel):
    """Key requirements extracted from a job description."""
    Required_Skills: List[str] = Field(description="Key skills, programming languages, tools or domain knowledge mentioned. Empty list if none.")
    Education_Required: str = Field(description="Minimum education level required/preferred (e.g. Bachelor's, MBA). 'Not specified' if not mentioned.")
    Experience_Required: str = Field(description="Years of experience or experience level required (e.g. '3+ years'). 'Not specified' if not mentioned.")

requirements_parser = PydanticOutputParser(pydantic_object=JobRequirements)

requirements_format_instructions = """Respond with ONLY a JSON object with exactly these keys, filled in with the actual values you found (not the schema, not descriptions):
{
  "Required_Skills": ["Python", "SQL"],
  "Education_Required": "Bachelor's degree",
  "Experience_Required": "3+ years"
}
Use "Not specified" for Education_Required or Experience_Required if not mentioned, and an empty list [] for Required_Skills if none are mentioned. Output nothing except this JSON object."""

requirements_template = ChatPromptTemplate([
    ("system", "You extract structured hiring requirements from job descriptions. Respond with valid JSON only, no extra commentary."),
    ("human", """Extract the required skills, education level, and years of experience from the job description below.
If a field is not mentioned, use "Not specified".

Job Title: {job_title}
Job Description: {job_description}

{format_instructions}"""),
]).partial(format_instructions=requirements_format_instructions)

requirements_chain = requirements_template | llm | requirements_parser


In [21]:
# Show this works for a sample datapoint
sample_requirements = to_model(
    requirements_chain.invoke({
        "job_title": sample_job_title,
        "job_description": sample_job_desc
    }),
    JobRequirements,
)
sample_requirements


JobRequirements(Required_Skills=[], Education_Required='Not specified', Experience_Required='Not specified')

## Step 4 & 5: Apply the LLM Chain to Each Job Posting and Update the DataFrame

In [22]:
job_results = []
for idx, row in jobs_df.iterrows():
    try:
        category = job_classification_chain.invoke({
            "categories": ", ".join(job_categories),
            "job_title": row["Job_Title"],
            "job_description": row["Job_Description"]
        }).strip()

        requirements = to_model(
            requirements_chain.invoke({
                "job_title": row["Job_Title"],
                "job_description": row["Job_Description"]
            }),
            JobRequirements,
        )

        job_results.append({
            "Predicted_Category": category,
            "Required_Skills": requirements.Required_Skills,
            "Education_Required": requirements.Education_Required,
            "Experience_Required": requirements.Experience_Required,
        })
    except Exception as e:
        print(f"Row {idx} failed: {e}")
        job_results.append({
            "Predicted_Category": "Not specified",
            "Required_Skills": [],
            "Education_Required": "Not specified",
            "Experience_Required": "Not specified",
        })

job_results_df = pd.DataFrame(job_results)
job_results_df.head()


,Predicted_Category,Required_Skills,Education_Required,Experience_Required
0,Technology/IT,[],Not specified,1 year (Preferred)
1,Technology/IT,"[Python, SQL, Django, flask, Linux, REST, RPC,...",Not specified,Not specified
2,Technology/IT,"[Python, Java]",Any Graduate or M.Sc.,3 years
3,Technology/IT,"[Objective-C, Cocoa Touch, Core Data, Core Ani...",Not specified,Not specified
4,Technology/IT,"[javascript, html, cs, restful apis, http, red...",Bachelor's degree,5+ years


In [23]:
# Final merged dataframe with all original and new columns together
jobs_final_df = pd.concat([jobs_df.reset_index(drop=True), job_results_df.reset_index(drop=True)], axis=1)
jobs_final_df.head()

,Job_Title,Job_Description,Predicted_Category,Required_Skills,Education_Required,Experience_Required
0,Flutter Developer,We are looking for hire experts flutter develo...,Technology/IT,[],Not specified,1 year (Preferred)
1,Django Developer,PYTHON/DJANGO (Developer/Lead) - Job Code(PDJ ...,Technology/IT,"[Python, SQL, Django, flask, Linux, REST, RPC,...",Not specified,Not specified
2,Machine Learning,"Data Scientist (Contractor)\r\n\r\nBangalore, ...",Technology/IT,"[Python, Java]",Any Graduate or M.Sc.,3 years
3,iOS Developer,JOB DESCRIPTION:\r\n\r\nStrong framework outsi...,Technology/IT,"[Objective-C, Cocoa Touch, Core Data, Core Ani...",Not specified,Not specified
4,Full Stack Developer,job responsibility full stack engineer – react...,Technology/IT,"[javascript, html, cs, restful apis, http, red...",Bachelor's degree,5+ years


In [24]:
jobs_final_df.to_csv("part2_job_analysis_results.csv", index=False)
jobs_final_df.shape

(25, 6)

---
## Notes
- Both parts use `llm.with_structured_output(...)` with a Pydantic schema so the model's output is validated/parsed automatically (same pattern as `3. Structured Output Generation.ipynb`).
- Each row is wrapped in a `try/except` so a single rate-limited/failed call doesn't stop the whole loop - failed rows fall back to `"Not specified"`.
- Bonus (running on the full dataset instead of the first 30/25 rows) was left out here due to time constraints - swap `model_provider="groq"` for a local Ollama model and remove the `.head(...)` calls to attempt it.